In [4]:
%pip install -U -q transformers datasets evaluate accelerate sentencepiece seqeval


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
from datasets import Dataset, DatasetDict, load_dataset
import numpy as np
import torch
from seqeval.metrics import accuracy_score, f1_score, precision_score, recall_score
from transformers import (
    AutoConfig,
    AutoModelForMaskedLM,
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    DataCollatorForTokenClassification,
    DataCollatorForWholeWordMask,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_callback import ProgressCallback
from transformers.utils.notebook import NotebookProgressCallback
import pandas as pd

/Users/user/my_projects/dl_nlp_2026/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_MODEL = "cointegrated/rubert-tiny2"

MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS = 3
MLM_EPOCHS = 5

## Загрузка и анализ данных


In [7]:
ds_raw = load_dataset("gusevski/factrueval2016")
ds_raw

Repo card metadata block was not found. Setting CardData to empty.


DatasetDict({
    train: Dataset({
        features: ['data'],
        num_rows: 1
    })
    validation: Dataset({
        features: ['data'],
        num_rows: 1
    })
    test: Dataset({
        features: ['data'],
        num_rows: 1
    })
})

## Подготовка данных


In [8]:
def unwrap_split(split_ds):
    rows = split_ds[0]["data"]
    return Dataset.from_list(rows)

ds = DatasetDict({
    "train": unwrap_split(ds_raw["train"]),
    "validation": unwrap_split(ds_raw["validation"]),
    "test": unwrap_split(ds_raw["test"]),
})
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags'],
        num_rows: 7746
    })
    validation: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags'],
        num_rows: 2582
    })
    test: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags'],
        num_rows: 2582
    })
})

In [9]:
print(ds["train"][2]["tokens"])
print(ds["train"][2]["ner_tags"])
print(ds["train"][2]["ner_tags_str"])

['В', 'Ханты-Мансийском', 'автономном', 'округе', 'с', 'должности', 'снят', 'начальник', 'УВД', 'Николай', 'Гудожников', '.']
[0, 5, 6, 6, 0, 0, 0, 0, 3, 1, 2, 0]
['O', 'B-LOC', 'I-LOC', 'I-LOC', 'O', 'O', 'O', 'O', 'B-ORG', 'B-PER', 'I-PER', 'O']


Датасет `gusevski/factrueval2016`: после разворачивания сплиты train/val/test — 7746 / 2582 / 2582 предложений; в примере видны токены и строковые NER-метки (`ner_tags_str`).


In [10]:
all_tags = {tag for name in ds for ex in ds[name] for tag in ex["ner_tags_str"]}
label_names = sorted(all_tags)
label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label for label, i in label2id.items()}
print(label_names)
print(label2id)
print(id2label)

['B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O']
{'B-LOC': 0, 'B-ORG': 1, 'B-PER': 2, 'I-LOC': 3, 'I-ORG': 4, 'I-PER': 5, 'O': 6}
{0: 'B-LOC', 1: 'B-ORG', 2: 'B-PER', 3: 'I-LOC', 4: 'I-ORG', 5: 'I-PER', 6: 'O'}


Построены `label2id` / `id2label` по полному множеству строковых NER-меток на всех сплитах — далее используется **len(label_names)** классов для головы классификации.


## Токенизация и выравнивание меток


In [11]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)


def tokenize_and_align_labels(example):
    t = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )
    word_ids = t.word_ids()
    ner = example["ner_tags_str"]
    previous_word_idx = None
    label_ids = []
    for word_idx in word_ids:
        if word_idx is None:
            label_ids.append(-100)
        elif word_idx != previous_word_idx:
            label_ids.append(label2id[ner[word_idx]])
        else:
            label_ids.append(-100)
        previous_word_idx = word_idx
    t["labels"] = label_ids
    return t

In [12]:
ds_tok = ds.map(
    tokenize_and_align_labels,
    remove_columns=ds["train"].column_names,
)
ds_tok

Map: 100%|██████████| 2582/2582 [00:00<00:00, 5795.01 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 7746
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2582
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2582
    })
})

In [13]:
ex = ds_tok["train"][2]
print("input_ids", ex["input_ids"][:10])
print("tokens", tokenizer.convert_ids_to_tokens(ex["input_ids"][:10]))
print("labels", ex["labels"][:10])

input_ids [2, 282, 22585, 17, 20859, 24542, 62962, 17122, 329, 12238]
tokens ['[CLS]', 'В', 'Ханты', '-', 'Манси', '##йском', 'автономном', 'округе', 'с', 'должности']
labels [-100, 6, 0, -100, -100, -100, 3, 3, 6, 6]


Для `rubert-tiny2` с `is_split_into_words=True` метки переносятся на первый субтокен слова, остальным ставится `-100`; в примере видно согласованность `input_ids`, piece-токенов и `labels`.


In [14]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    true_preds, true_labels = [], []
    for pred_row, lab_row in zip(predictions, labels):
        pr_tags, lb_tags = [], []
        for p, lb in zip(pred_row, lab_row):
            if lb == -100:
                continue
            pr_tags.append(id2label[int(p)])
            lb_tags.append(id2label[int(lb)])
        true_preds.append(pr_tags)
        true_labels.append(lb_tags)
    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
        "accuracy": accuracy_score(true_labels, true_preds),
    }

In [15]:
def swap_notebook_progress_callback(trainer):
    for cb in list(trainer.callback_handler.callbacks):
        if isinstance(cb, NotebookProgressCallback):
            trainer.remove_callback(cb)
            trainer.add_callback(ProgressCallback())
            break

In [16]:
def ner_metrics_from_evals(before_eval, after_eval):
    return {
        "before_f1": before_eval["eval_f1"],
        "after_f1": after_eval["eval_f1"],
        "before_precision": before_eval["eval_precision"],
        "after_precision": after_eval["eval_precision"],
        "before_recall": before_eval["eval_recall"],
        "after_recall": after_eval["eval_recall"],
        "before_accuracy": before_eval["eval_accuracy"],
        "after_accuracy": after_eval["eval_accuracy"],
    }

In [17]:
def ner_training_args(checkpoint_dir):
    return TrainingArguments(
        output_dir=checkpoint_dir,
        learning_rate=2e-5,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        fp16=torch.cuda.is_available(),
        report_to="none",
    )

## Baseline модель


In [18]:
BASELINE_NER_DIR = "./hw3_ner_artifacts/baseline_ner"

config = AutoConfig.from_pretrained(BASE_MODEL)
config.num_labels = len(label2id)
config.id2label = id2label
config.label2id = label2id

model = AutoModelForTokenClassification.from_pretrained(BASE_MODEL, config=config)

training_args = ner_training_args(f"{BASELINE_NER_DIR}/trainer_checkpoints")
data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
swap_notebook_progress_callback(trainer)
trainer

Loading weights: 100%|██████████| 53/53 [00:00<00:00, 14481.02it/s]
BertForTokenClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignore

In [19]:
before_eval = trainer.evaluate()
trainer.train()
after_eval = trainer.evaluate()

ner_metrics = ner_metrics_from_evals(before_eval, after_eval)
ner_metrics

/Users/user/my_projects/dl_nlp_2026/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 162/162 [00:03<00:00, 43.69it/s]
                                                  
 33%|███▎      | 485/1455 [00:25<00:48, 20.19it/s]

{'eval_loss': '0.1341', 'eval_model_preparation_time': '0.0005', 'eval_precision': '0.7394', 'eval_recall': '0.821', 'eval_f1': '0.7781', 'eval_accuracy': '0.9643', 'eval_runtime': '1.773', 'eval_samples_per_second': '1456', 'eval_steps_per_second': '91.36', 'epoch': '1'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 15.06it/s]
/Users/user/my_projects/dl_nlp_2026/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
 35%|███▍      | 505/1455 [00:26<01:00, 15.61it/s]

{'loss': '0.3552', 'grad_norm': '0.9958', 'learning_rate': '1.314e-05', 'epoch': '1.031'}


                                                  
 67%|██████▋   | 970/1455 [00:47<00:15, 31.46it/s]

...

{'eval_loss': '0.0818', 'eval_model_preparation_time': '0.0005', 'eval_precision': '0.8205', 'eval_recall': '0.8833', 'eval_f1': '0.8508', 'eval_accuracy': '0.9756', 'eval_runtime': '1.538', 'eval_samples_per_second': '1679', 'eval_steps_per_second': '105.3', 'epoch': '3'}


100%|██████████| 1455/1455 [01:08<00:00, 31.94it/s]There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encode

{'train_runtime': '68.05', 'train_samples_per_second': '341.5', 'train_steps_per_second': '21.38', 'train_loss': '0.1878', 'epoch': '3'}


100%|██████████| 162/162 [00:01<00:00, 88.52it/s] 


{'before_f1': 0.026586372253819556,
 'after_f1': 0.8507807447103389,
 'before_precision': 0.015041768669712036,
 'after_precision': 0.8205310996257351,
 'before_recall': 0.11435149654643131,
 'after_recall': 0.8833461243284727,
 'before_accuracy': 0.0692962919548314,
 'after_accuracy': 0.9755987965459305}

До обучения на случайной инициализации головы F1 на val ≈ 0.027; после 3 эпох best checkpoint даёт **F1 ≈ 0.851**, accuracy токенов ≈ 0.976 (рост за счёт дообучения всей модели под NER).


In [20]:
trainer.save_model(BASELINE_NER_DIR)
tokenizer.save_pretrained(BASELINE_NER_DIR)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 19.65it/s]


('./hw3_ner_artifacts/baseline_ner/tokenizer_config.json',
 './hw3_ner_artifacts/baseline_ner/tokenizer.json')

## MLM pretraining


In [21]:
mlm_model = AutoModelForMaskedLM.from_pretrained(BASE_MODEL)
mlm_model

Loading weights: 100%|██████████| 58/58 [00:00<00:00, 12111.40it/s]
BertForMaskedLM LOAD REPORT from: cointegrated/rubert-tiny2
Key                          | Status     |  | 
-----------------------------+------------+--+-
cls.seq_relationship.bias    | UNEXPECTED |  | 
cls.seq_relationship.weight  | UNEXPECTED |  | 
bert.pooler.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
bert.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(83828, 312, padding_idx=0)
      (position_embeddings): Embedding(2048, 312)
      (token_type_embeddings): Embedding(2, 312)
      (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-2): 3 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=312, out_features=312, bias=True)
              (key): Linear(in_features=312, out_features=312, bias=True)
              (value): Linear(in_features=312, out_features=312, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=312, out_features=312, bias=True)
              (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_af

In [22]:
mlm_text_rows_train = [{"text_for_mlm": " ".join(ex["tokens"])} for ex in ds["train"]]
mlm_text_rows_val = [{"text_for_mlm": " ".join(ex["tokens"])} for ex in ds["validation"]]
mlm_raw = Dataset.from_list(mlm_text_rows_train)
mlm_raw_val = Dataset.from_list(mlm_text_rows_val)
mlm_raw, mlm_raw_val

(Dataset({
     features: ['text_for_mlm'],
     num_rows: 7746
 }),
 Dataset({
     features: ['text_for_mlm'],
     num_rows: 2582
 }))

In [23]:
def tokenize_for_mlm(examples):
    return tokenizer(
        examples["text_for_mlm"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_special_tokens_mask=True,
        return_offsets_mapping=True,
    )


mlm_ds = mlm_raw.map(
    tokenize_for_mlm,
    batched=True,
    remove_columns=mlm_raw.column_names,
)
mlm_ds_val = mlm_raw_val.map(
    tokenize_for_mlm,
    batched=True,
    remove_columns=mlm_raw_val.column_names,
)

mlm_ds, mlm_ds_val

Map: 100%|██████████| 2582/2582 [00:00<00:00, 6081.85 examples/s]


(Dataset({
     features: ['input_ids', 'token_type_ids', 'attention_mask', 'special_tokens_mask', 'offset_mapping'],
     num_rows: 7746
 }),
 Dataset({
     features: ['input_ids', 'token_type_ids', 'attention_mask', 'special_tokens_mask', 'offset_mapping'],
     num_rows: 2582
 }))

In [24]:
mlm_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)
mlm_collator

DataCollatorForLanguageModeling(tokenizer=BertTokenizer(name_or_path='cointegrated/rubert-tiny2', vocab_size=83828, model_max_length=2048, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}), mlm=True, whole_word_mask=False, mlm_probability=0.15, mask_replace_prob=0.8, random_replace_prob=0.1, pad_to_multiple_of=None, return_te

In [25]:
def mlm_training_args(checkpoint_dir):
    return TrainingArguments(
        output_dir=checkpoint_dir,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=5e-5,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        fp16=torch.cuda.is_available(),
        report_to="none",
        remove_unused_columns=False,
    )

In [26]:
MLM_RANDOM_MASKING_DIR = "./hw3_ner_artifacts/mlm_random_masking"

mlm_training_args_config = mlm_training_args(f"{MLM_RANDOM_MASKING_DIR}/trainer_checkpoints")
mlm_training_args_config

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=Fals

In [27]:
mlm_trainer = Trainer(
    model=mlm_model,
    args=mlm_training_args_config,
    train_dataset=mlm_ds,
    eval_dataset=mlm_ds_val,
    data_collator=mlm_collator,
    processing_class=tokenizer,
)

swap_notebook_progress_callback(mlm_trainer)

metrics_mlm_before = mlm_trainer.evaluate()
mlm_trainer.train()
metrics_mlm_after = mlm_trainer.evaluate()

 33%|███▎      | 485/1455 [03:04<06:06,  2.65it/s]

{'eval_loss': '3.263', 'eval_model_preparation_time': '0.0003', 'eval_runtime': '16.16', 'eval_samples_per_second': '159.8', 'eval_steps_per_second': '10.02', 'epoch': '1'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 20.91it/s]
/Users/user/my_projects/dl_nlp_2026/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
 34%|███▍      | 500/1455 [03:10<06:06,  2.61it/s]  

{'loss': '3.35', 'grad_norm': '14.67', 'learning_rate': '3.285e-05', 'epoch': '1.031'}


 67%|██████▋   | 970/1455 [06:08<02:28,  3.27it/s]

...

100%|██████████| 1455/1455 [09:08<00:00,  3.19it/s]

{'eval_loss': '3.188', 'eval_model_preparation_time': '0.0003', 'eval_runtime': '16.11', 'eval_samples_per_second': '160.3', 'eval_steps_per_second': '10.06', 'epoch': '3'}


100%|██████████| 1455/1455 [09:08<00:00,  3.19it/s]There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.decoder.weight', 'cls.predictions.decoder.bias'].
There were unexpected keys in the checkpo

{'train_runtime': '548.9', 'train_samples_per_second': '42.34', 'train_steps_per_second': '2.651', 'train_loss': '3.177', 'epoch': '3'}


100%|██████████| 162/162 [00:16<00:00,  9.96it/s]


In [28]:
mlm_run_metrics = {
    "mlm_loss_before": metrics_mlm_before["eval_loss"],
    "mlm_loss_after": metrics_mlm_after["eval_loss"],
}
mlm_run_metrics

{'mlm_loss_before': 3.5802831649780273, 'mlm_loss_after': 3.170499563217163}

In [29]:
mlm_trainer.save_model(MLM_RANDOM_MASKING_DIR)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  7.52it/s]


Дополнительное MLM на текстах train (случайное маскирование 15%): eval loss снизился с **≈ 3.58 до ≈ 3.17** за 3 эпохи; веса сохранены в `mlm_random_masking`.


## MLM + NER fine-tuning


In [30]:
ner_config_mlm = AutoConfig.from_pretrained(MLM_RANDOM_MASKING_DIR)
ner_config_mlm.num_labels = len(label2id)
ner_config_mlm.id2label = id2label
ner_config_mlm.label2id = label2id

model_mlm_ner = AutoModelForTokenClassification.from_pretrained(
    MLM_RANDOM_MASKING_DIR,
    config=ner_config_mlm,
)
model_mlm_ner

Loading weights: 100%|██████████| 53/53 [00:00<00:00, 12247.83it/s]
BertForTokenClassification LOAD REPORT from: ./hw3_ner_artifacts/mlm_random_masking
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(83828, 312, padding_idx=0)
      (position_embeddings): Embedding(2048, 312)
      (token_type_embeddings): Embedding(2, 312)
      (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-2): 3 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=312, out_features=312, bias=True)
              (key): Linear(in_features=312, out_features=312, bias=True)
              (value): Linear(in_features=312, out_features=312, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=312, out_features=312, bias=True)
              (LayerNorm): LayerNorm((312,), eps=1e-12, ele

In [31]:
NER_MLM_DIR = "./hw3_ner_artifacts/mlm_ner"

ner_mlm_training_args = ner_training_args(f"{NER_MLM_DIR}/trainer_checkpoints")
ner_mlm_training_args

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=Fals

In [32]:
ner_mlm_collator = DataCollatorForTokenClassification(tokenizer)

trainer_mlm_ner = Trainer(
    model=model_mlm_ner,
    args=ner_mlm_training_args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    data_collator=ner_mlm_collator,
    compute_metrics=compute_metrics,
)
swap_notebook_progress_callback(trainer_mlm_ner)
trainer_mlm_ner

In [34]:
before_eval_mlm = trainer_mlm_ner.evaluate()
trainer_mlm_ner.train()
after_eval_mlm = trainer_mlm_ner.evaluate()

ner_metrics_mlm = ner_metrics_from_evals(before_eval_mlm, after_eval_mlm)
ner_metrics_mlm

 33%|███▎      | 485/1455 [00:17<00:31, 31.08it/s]

{'eval_loss': '0.1144', 'eval_model_preparation_time': '0.0009', 'eval_precision': '0.7886', 'eval_recall': '0.8425', 'eval_f1': '0.8147', 'eval_accuracy': '0.9692', 'eval_runtime': '1.516', 'eval_samples_per_second': '1703', 'eval_steps_per_second': '106.9', 'epoch': '1'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 16.29it/s]
/Users/user/my_projects/dl_nlp_2026/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
 35%|███▍      | 505/1455 [00:18<01:02, 15.30it/s]

{'loss': '0.3301', 'grad_norm': '1.169', 'learning_rate': '1.314e-05', 'epoch': '1.031'}


 67%|██████▋   | 970/1455 [00:33<00:12, 38.17it/s]

...

{'eval_loss': '0.07459', 'eval_model_preparation_time': '0.0009', 'eval_precision': '0.8531', 'eval_recall': '0.8914', 'eval_f1': '0.8718', 'eval_accuracy': '0.978', 'eval_runtime': '1.386', 'eval_samples_per_second': '1863', 'eval_steps_per_second': '116.9', 'epoch': '3'}


100%|██████████| 1455/1455 [00:49<00:00, 35.48it/s]There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encode

{'train_runtime': '49.01', 'train_samples_per_second': '474.1', 'train_steps_per_second': '29.69', 'train_loss': '0.1709', 'epoch': '3'}


100%|██████████| 162/162 [00:01<00:00, 113.06it/s]


{'before_f1': 0.037565905096660815,
 'after_f1': 0.8718333646087447,
 'before_precision': 0.021211670139922596,
 'after_precision': 0.8531031950055087,
 'before_recall': 0.1640445126630852,
 'after_recall': 0.8914044512663085,
 'before_accuracy': 0.09844488727386395,
 'after_accuracy': 0.9779627241823936}

In [35]:
{
    "baseline_after_f1": ner_metrics["after_f1"],
    "mlm_pretrained_after_f1": ner_metrics_mlm["after_f1"],
    "delta_f1": ner_metrics_mlm["after_f1"] - ner_metrics["after_f1"],
}

{'baseline_after_f1': 0.8507807447103389,
 'mlm_pretrained_after_f1': 0.8718333646087447,
 'delta_f1': 0.02105261989840579}

NER поверх MLM-весов: F1 на val после fine-tuning **≈ 0.872** против **≈ 0.851** у baseline — прирост **≈ +0.021** по seqeval F1 при сопоставимых 3 эпохах.


## Улучшения MLM (Whole Word Masking)


In [36]:
mlm_whole_word_collator = DataCollatorForWholeWordMask(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)
mlm_whole_word_collator

/Users/user/my_projects/dl_nlp_2026/venv/lib/python3.12/site-packages/transformers/data/data_collator.py:1028: FutureWarning: DataCollatorForWholeWordMask is deprecated and will be removed in a future version, you can now use DataCollatorForLanguageModeling with whole_word_mask=True instead.
  warnings.warn(


DataCollatorForWholeWordMask(tokenizer=BertTokenizer(name_or_path='cointegrated/rubert-tiny2', vocab_size=83828, model_max_length=2048, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}), mlm=True, whole_word_mask=True, mlm_probability=0.15, mask_replace_prob=0.8, random_replace_prob=0.1, pad_to_multiple_of=None, return_tensor

In [37]:
MLM_WWM_DIR = "./hw3_ner_artifacts/mlm_whole_word"

mlm_wwm_training_args = mlm_training_args(
    f"{MLM_WWM_DIR}/trainer_checkpoints",
)

In [38]:
mlm_wwm_trainer = Trainer(
    model=mlm_model,
    args=mlm_wwm_training_args,
    train_dataset=mlm_ds,
    eval_dataset=mlm_ds_val,
    data_collator=mlm_whole_word_collator,
    processing_class=tokenizer,
)

swap_notebook_progress_callback(mlm_wwm_trainer)

metrics_mlm_wwm_before = mlm_wwm_trainer.evaluate()
mlm_wwm_trainer.train()
metrics_mlm_wwm_after = mlm_wwm_trainer.evaluate()

 33%|███▎      | 485/1455 [03:10<05:07,  3.15it/s]

{'eval_loss': '4.051', 'eval_model_preparation_time': '0.0004', 'eval_runtime': '20.05', 'eval_samples_per_second': '128.8', 'eval_steps_per_second': '8.08', 'epoch': '1'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 10.03it/s]
/Users/user/my_projects/dl_nlp_2026/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
 34%|███▍      | 500/1455 [03:18<07:13,  2.21it/s]  

{'loss': '3.494', 'grad_norm': '15.4', 'learning_rate': '3.285e-05', 'epoch': '1.031'}


 67%|██████▋   | 970/1455 [06:23<02:44,  2.95it/s]

...

100%|██████████| 1455/1455 [09:22<00:00,  2.94it/s]

{'eval_loss': '4.067', 'eval_model_preparation_time': '0.0004', 'eval_runtime': '16.78', 'eval_samples_per_second': '153.9', 'eval_steps_per_second': '9.654', 'epoch': '3'}


100%|██████████| 1455/1455 [09:22<00:00,  2.94it/s]There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.decoder.weight', 'cls.predictions.decoder.bias'].
There were unexpected keys in the checkpo

{'train_runtime': '563', 'train_samples_per_second': '41.28', 'train_steps_per_second': '2.584', 'train_loss': '3.576', 'epoch': '3'}


100%|██████████| 162/162 [00:16<00:00,  9.88it/s]


In [40]:
mlm_wwm_metrics = {
    "mlm_loss_before": metrics_mlm_wwm_before["eval_loss"],
    "mlm_loss_after": metrics_mlm_wwm_after["eval_loss"],
}
mlm_wwm_metrics

{'mlm_loss_before': 4.273593902587891, 'mlm_loss_after': 4.0600504875183105}

In [41]:
mlm_wwm_trainer.save_model(MLM_WWM_DIR)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.23it/s]


MLM с `DataCollatorForWholeWordMask`: eval loss **≈ 4.27 → 4.06** (абсолютные значения выше, чем у посимвольного MLM — ожидаемо из-за другой постановки задачи); чекпоинт в `mlm_whole_word`.


In [42]:
ner_config_mlm_wwm = AutoConfig.from_pretrained(MLM_WWM_DIR)
ner_config_mlm_wwm.num_labels = len(label2id)
ner_config_mlm_wwm.id2label = id2label
ner_config_mlm_wwm.label2id = label2id

In [43]:
model_mlm_wwm_ner = AutoModelForTokenClassification.from_pretrained(
    MLM_WWM_DIR,
    config=ner_config_mlm_wwm,
)
model_mlm_wwm_ner

Loading weights: 100%|██████████| 53/53 [00:00<00:00, 14114.17it/s]
BertForTokenClassification LOAD REPORT from: ./hw3_ner_artifacts/mlm_whole_word
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(83828, 312, padding_idx=0)
      (position_embeddings): Embedding(2048, 312)
      (token_type_embeddings): Embedding(2, 312)
      (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-2): 3 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=312, out_features=312, bias=True)
              (key): Linear(in_features=312, out_features=312, bias=True)
              (value): Linear(in_features=312, out_features=312, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=312, out_features=312, bias=True)
              (LayerNorm): LayerNorm((312,), eps=1e-12, ele

In [44]:
NER_MLM_WWM_DIR = "./hw3_ner_artifacts/mlm_wwm_ner"

ner_mlm_wwm_training_args = ner_training_args(f"{NER_MLM_WWM_DIR}/trainer_checkpoints")
ner_mlm_wwm_training_args

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=Fals

In [45]:
ner_mlm_wwm_collator = DataCollatorForTokenClassification(tokenizer)

trainer_mlm_wwm_ner = Trainer(
    model=model_mlm_wwm_ner,
    args=ner_mlm_wwm_training_args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    data_collator=ner_mlm_wwm_collator,
    compute_metrics=compute_metrics,
)
swap_notebook_progress_callback(trainer_mlm_wwm_ner)
trainer_mlm_wwm_ner

In [46]:
before_eval_mlm_wwm = trainer_mlm_wwm_ner.evaluate()
trainer_mlm_wwm_ner.train()
after_eval_mlm_wwm = trainer_mlm_wwm_ner.evaluate()

  0%|          | 0/162 [00:00<?, ?it/s]

 33%|███▎      | 485/1455 [00:15<00:28, 33.78it/s]

{'eval_loss': '0.1195', 'eval_model_preparation_time': '0.0008', 'eval_precision': '0.7843', 'eval_recall': '0.8273', 'eval_f1': '0.8052', 'eval_accuracy': '0.9674', 'eval_runtime': '1.361', 'eval_samples_per_second': '1898', 'eval_steps_per_second': '119.1', 'epoch': '1'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 16.85it/s]
/Users/user/my_projects/dl_nlp_2026/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
 35%|███▍      | 507/1455 [00:16<00:48, 19.71it/s]

{'loss': '0.3319', 'grad_norm': '1.099', 'learning_rate': '1.314e-05', 'epoch': '1.031'}


...

100%|██████████| 1455/1455 [00:47<00:00, 33.66it/s]

{'eval_loss': '0.07859', 'eval_model_preparation_time': '0.0008', 'eval_precision': '0.8474', 'eval_recall': '0.883', 'eval_f1': '0.8648', 'eval_accuracy': '0.9766', 'eval_runtime': '1.407', 'eval_samples_per_second': '1835', 'eval_steps_per_second': '115.1', 'epoch': '3'}


100%|██████████| 1455/1455 [00:47<00:00, 33.66it/s]There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encode

{'train_runtime': '47.79', 'train_samples_per_second': '486.3', 'train_steps_per_second': '30.45', 'train_loss': '0.1723', 'epoch': '3'}


100%|██████████| 162/162 [00:01<00:00, 94.00it/s] 


In [47]:
ner_metrics_mlm_wwm = ner_metrics_from_evals(before_eval_mlm_wwm, after_eval_mlm_wwm)
ner_metrics_mlm_wwm

{'before_f1': 0.03740929373056295,
 'after_f1': 0.8647937611575683,
 'before_precision': 0.02119401460599561,
 'after_precision': 0.8473577610016572,
 'before_recall': 0.15924788948580199,
 'after_recall': 0.8829623944742901,
 'before_accuracy': 0.11749306450982691,
 'after_accuracy': 0.9765951627398116}

In [49]:
{
    "baseline_after_f1": ner_metrics["after_f1"],
    "mlm_pretrained_after_f1": ner_metrics_mlm_wwm["after_f1"],
    "delta_f1": ner_metrics_mlm_wwm["after_f1"] - ner_metrics["after_f1"],
}

{'baseline_after_f1': 0.8507807447103389,
 'mlm_pretrained_after_f1': 0.8647937611575683,
 'delta_f1': 0.01401301644722941}

NER после WWM-MLM: F1 **≈ 0.865** — выше baseline (+≈0.014), но **ниже**, чем после обычного MLM (**≈ 0.872**): в этом прогоне whole word masking не дал дополнительного выигрыша над случайным маскированием субтокенов.
